In [3]:
# Simple MCP (Model Context Protocol) Implementation for Stock Analysis
# This notebook contains a basic MCP server with tools for financial data analysis
# !pip install yfinance
import json
import asyncio
from typing import Any, Dict, List, Optional
from dataclasses import dataclass
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

print("MCP Stock Analysis Tools - Initialized")


MCP Stock Analysis Tools - Initialized


In [4]:
# MCP Tool Definitions
@dataclass
class MCPTool:
    """Base class for MCP tools"""
    name: str
    description: str
    input_schema: Dict[str, Any]
    
    async def execute(self, **kwargs) -> Dict[str, Any]:
        """Execute the tool with given parameters"""
        raise NotImplementedError

class StockDataTool(MCPTool):
    """Tool for fetching stock data"""
    
    def __init__(self):
        super().__init__(
            name="get_stock_data",
            description="Fetch stock data for a given symbol",
            input_schema={
                "type": "object",
                "properties": {
                    "symbol": {"type": "string", "description": "Stock symbol (e.g., AAPL, GOOGL)"},
                    "period": {"type": "string", "description": "Time period (1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max)", "default": "1mo"},
                    "interval": {"type": "string", "description": "Data interval (1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 1d, 5d, 1wk, 1mo, 3mo)", "default": "1d"}
                },
                "required": ["symbol"]
            }
        )
    
    async def execute(self, symbol: str, period: str = "1mo", interval: str = "1d") -> Dict[str, Any]:
        try:
            ticker = yf.Ticker(symbol)
            data = ticker.history(period=period, interval=interval)
            
            if data.empty:
                return {"error": f"No data found for symbol {symbol}"}
            
            # Convert to JSON-serializable format
            result = {
                "symbol": symbol,
                "period": period,
                "interval": interval,
                "data_points": len(data),
                "latest_price": float(data['Close'].iloc[-1]),
                "price_change": float(data['Close'].iloc[-1] - data['Open'].iloc[0]),
                "price_change_percent": float((data['Close'].iloc[-1] - data['Open'].iloc[0]) / data['Open'].iloc[0] * 100),
                "data": data.to_dict('records')
            }
            
            return {"success": True, "result": result}
            
        except Exception as e:
            return {"error": f"Failed to fetch stock data: {str(e)}"}

print("MCP Tool classes defined successfully")


MCP Tool classes defined successfully


In [6]:
class StockInfoTool(MCPTool):
    """Tool for fetching stock information"""
    
    def __init__(self):
        super().__init__(
            name="get_stock_info",
            description="Get detailed information about a stock",
            input_schema={
                "type": "object",
                "properties": {
                    "symbol": {"type": "string", "description": "Stock symbol (e.g., AAPL, GOOGL)"}
                },
                "required": ["symbol"]
            }
        )
    
    async def execute(self, symbol: str) -> Dict[str, Any]:
        try:
            ticker = yf.Ticker(symbol)
            info = ticker.info
            
            # Extract key information
            result = {
                "symbol": symbol,
                "name": info.get("longName", "N/A"),
                "sector": info.get("sector", "N/A"),
                "industry": info.get("industry", "N/A"),
                "market_cap": info.get("marketCap", "N/A"),
                "current_price": info.get("currentPrice", info.get("regularMarketPrice", "N/A")),
                "pe_ratio": info.get("trailingPE", "N/A"),
                "dividend_yield": info.get("dividendYield", "N/A"),
                "52_week_high": info.get("fiftyTwoWeekHigh", "N/A"),
                "52_week_low": info.get("fiftyTwoWeekLow", "N/A"),
                "volume": info.get("volume", "N/A"),
                "average_volume": info.get("averageVolume", "N/A"),
                "description": info.get("longBusinessSummary", "N/A")
            }
            
            return {"success": True, "result": result}
            
        except Exception as e:
            return {"error": f"Failed to fetch stock info: {str(e)}"}

class TechnicalAnalysisTool(MCPTool):
    """Tool for basic technical analysis"""
    
    def __init__(self):
        super().__init__(
            name="technical_analysis",
            description="Perform basic technical analysis on stock data",
            input_schema={
                "type": "object",
                "properties": {
                    "symbol": {"type": "string", "description": "Stock symbol"},
                    "period": {"type": "string", "description": "Time period for analysis", "default": "3mo"}
                },
                "required": ["symbol"]
            }
        )
    
    async def execute(self, symbol: str, period: str = "3mo") -> Dict[str, Any]:
        try:
            ticker = yf.Ticker(symbol)
            data = ticker.history(period=period)
            
            if data.empty:
                return {"error": f"No data found for symbol {symbol}"}
            
            # Calculate basic technical indicators
            closes = data['Close']
            highs = data['High']
            lows = data['Low']
            
            # Moving averages
            ma_20 = closes.rolling(window=20).mean().iloc[-1]
            ma_50 = closes.rolling(window=50).mean().iloc[-1]
            
            # RSI calculation (simplified)
            delta = closes.diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            rs = gain / loss
            rsi = 100 - (100 / (1 + rs))
            current_rsi = rsi.iloc[-1]
            
            # Support and resistance levels
            resistance = highs.rolling(window=20).max().iloc[-1]
            support = lows.rolling(window=20).min().iloc[-1]
            
            result = {
                "symbol": symbol,
                "period": period,
                "current_price": float(closes.iloc[-1]),
                "moving_average_20": float(ma_20) if not pd.isna(ma_20) else None,
                "moving_average_50": float(ma_50) if not pd.isna(ma_50) else None,
                "rsi": float(current_rsi) if not pd.isna(current_rsi) else None,
                "resistance_level": float(resistance),
                "support_level": float(support),
                "trend": "Bullish" if closes.iloc[-1] > ma_20 else "Bearish" if not pd.isna(ma_20) else "Neutral"
            }
            
            return {"success": True, "result": result}
            
        except Exception as e:
            return {"error": f"Failed to perform technical analysis: {str(e)}"}

print("Additional MCP tools defined successfully")


Additional MCP tools defined successfully


In [7]:
# MCP Server Implementation
class MCPServer:
    """Simple MCP Server for Stock Analysis"""
    
    def __init__(self):
        self.tools = {
            "get_stock_data": StockDataTool(),
            "get_stock_info": StockInfoTool(),
            "technical_analysis": TechnicalAnalysisTool()
        }
    
    def list_tools(self) -> Dict[str, Any]:
        """List all available tools"""
        tools_list = []
        for tool_name, tool in self.tools.items():
            tools_list.append({
                "name": tool.name,
                "description": tool.description,
                "input_schema": tool.input_schema
            })
        
        return {
            "jsonrpc": "2.0",
            "id": 1,
            "result": {
                "tools": tools_list
            }
        }
    
    async def call_tool(self, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        """Call a specific tool with arguments"""
        if tool_name not in self.tools:
            return {
                "jsonrpc": "2.0",
                "id": 1,
                "error": {
                    "code": -32601,
                    "message": f"Tool '{tool_name}' not found"
                }
            }
        
        try:
            tool = self.tools[tool_name]
            result = await tool.execute(**arguments)
            
            return {
                "jsonrpc": "2.0",
                "id": 1,
                "result": {
                    "content": [
                        {
                            "type": "text",
                            "text": json.dumps(result, indent=2)
                        }
                    ]
                }
            }
        except Exception as e:
            return {
                "jsonrpc": "2.0",
                "id": 1,
                "error": {
                    "code": -32603,
                    "message": f"Internal error: {str(e)}"
                }
            }

# Initialize the MCP Server
mcp_server = MCPServer()
print("MCP Server initialized with tools:")
for tool_name in mcp_server.tools.keys():
    print(f"- {tool_name}")


MCP Server initialized with tools:
- get_stock_data
- get_stock_info
- technical_analysis


In [8]:
# Example Usage and Testing
async def test_mcp_tools():
    """Test the MCP tools with sample data"""
    print("Testing MCP Tools...")
    print("=" * 50)
    
    # Test 1: Get stock info
    print("1. Testing get_stock_info for AAPL:")
    result1 = await mcp_server.call_tool("get_stock_info", {"symbol": "AAPL"})
    print(json.dumps(result1, indent=2)[:500] + "...")
    print()
    
    # Test 2: Get stock data
    print("2. Testing get_stock_data for GOOGL:")
    result2 = await mcp_server.call_tool("get_stock_data", {"symbol": "GOOGL", "period": "5d"})
    print(json.dumps(result2, indent=2)[:500] + "...")
    print()
    
    # Test 3: Technical analysis
    print("3. Testing technical_analysis for MSFT:")
    result3 = await mcp_server.call_tool("technical_analysis", {"symbol": "MSFT", "period": "3mo"})
    print(json.dumps(result3, indent=2)[:500] + "...")
    print()
    
    # Test 4: List all tools
    print("4. Available tools:")
    tools_result = mcp_server.list_tools()
    for tool in tools_result["result"]["tools"]:
        print(f"- {tool['name']}: {tool['description']}")

# Run the test
await test_mcp_tools()


Testing MCP Tools...
1. Testing get_stock_info for AAPL:
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "{\n  \"success\": true,\n  \"result\": {\n    \"symbol\": \"AAPL\",\n    \"name\": \"Apple Inc.\",\n    \"sector\": \"Technology\",\n    \"industry\": \"Consumer Electronics\",\n    \"market_cap\": 3473692426240,\n    \"current_price\": 234.07,\n    \"pe_ratio\": 35.465153,\n    \"dividend_yield\": 0.44,\n    \"52_week_high\": 260.1,\n    \"52_week_low\": 169.21,\n    \"volume\": 5522...

2. Testing get_stock_data for GOOGL:
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "{\n  \"success\": true,\n  \"result\": {\n    \"symbol\": \"GOOGL\",\n    \"period\": \"5d\",\n    \"interval\": \"1d\",\n    \"data_points\": 5,\n    \"latest_price\": 240.8000030517578,\n    \"price_change\": 5.3300018310546875,\n    \"price_change_percent\": 2.263558756284603,\n    \"

In [9]:
# Integration with existing RAG system
class MCPRAGIntegration:
    """Integration layer between MCP tools and existing RAG system"""
    
    def __init__(self, mcp_server: MCPServer):
        self.mcp_server = mcp_server
    
    async def enhanced_query(self, query: str) -> Dict[str, Any]:
        """
        Enhanced query processing that can use MCP tools
        when stock-related queries are detected
        """
        query_lower = query.lower()
        
        # Check if query is about specific stock data
        if any(keyword in query_lower for keyword in ['stock', 'price', 'ticker', 'symbol']):
            # Extract potential stock symbols (simple heuristic)
            words = query.split()
            potential_symbols = [word.upper() for word in words if len(word) <= 5 and word.isalpha()]
            
            results = []
            for symbol in potential_symbols[:2]:  # Limit to 2 symbols
                try:
                    # Get stock info
                    info_result = await self.mcp_server.call_tool("get_stock_info", {"symbol": symbol})
                    if "error" not in info_result:
                        results.append({
                            "type": "stock_info",
                            "symbol": symbol,
                            "data": info_result
                        })
                    
                    # Get current price data
                    data_result = await self.mcp_server.call_tool("get_stock_data", {"symbol": symbol, "period": "5d"})
                    if "error" not in data_result:
                        results.append({
                            "type": "stock_data",
                            "symbol": symbol,
                            "data": data_result
                        })
                        
                except Exception as e:
                    continue
            
            return {
                "query": query,
                "enhanced_results": results,
                "mcp_enhanced": True
            }
        
        return {
            "query": query,
            "mcp_enhanced": False,
            "message": "Query doesn't appear to be stock-related"
        }

# Initialize the integration
mcp_rag = MCPRAGIntegration(mcp_server)
print("MCP-RAG Integration layer initialized")


MCP-RAG Integration layer initialized


In [ ]:
# Test the MCP-RAG integration
async def test_integration():
    """Test the integration with sample queries"""
    print("Testing MCP-RAG Integration...")
    print("=" * 50)
    
    # Test queries
    test_queries = [
        "What is the current price of AAPL?",
        "Show me information about Tesla stock",
        "How is the market doing today?",
        "What's the technical analysis for GOOGL?",
        "Tell me about Microsoft"
    ]
    
    for query in test_queries:
        print(f"Query: {query}")
        result = await mcp_rag.enhanced_query(query)
        print(f"MCP Enhanced: {result['mcp_enhanced']}")
        if result['mcp_enhanced']:
            print(f"Found {len(result['enhanced_results'])} enhanced results")
        else:
            print(result['message'])
        print("-" * 30)

# Run integration test
await test_integration()


# MCP Stock Analysis Tools - Usage Guide

## Overview
This notebook implements a simple Model Context Protocol (MCP) server with tools for stock analysis and financial data retrieval.

## Available Tools

### 1. `get_stock_data`
- **Purpose**: Fetch historical stock data
- **Parameters**:
  - `symbol` (required): Stock symbol (e.g., AAPL, GOOGL)
  - `period` (optional): Time period (1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max)
  - `interval` (optional): Data interval (1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 1d, 5d, 1wk, 1mo, 3mo)

### 2. `get_stock_info`
- **Purpose**: Get detailed company information
- **Parameters**:
  - `symbol` (required): Stock symbol

### 3. `technical_analysis`
- **Purpose**: Perform basic technical analysis
- **Parameters**:
  - `symbol` (required): Stock symbol
  - `period` (optional): Time period for analysis

## Usage Examples

```python
# Get stock information
result = await mcp_server.call_tool("get_stock_info", {"symbol": "AAPL"})

# Get stock data
result = await mcp_server.call_tool("get_stock_data", {"symbol": "GOOGL", "period": "1mo"})

# Technical analysis
result = await mcp_server.call_tool("technical_analysis", {"symbol": "MSFT"})
```

## Integration with RAG
The MCP tools are integrated with your existing RAG system to provide enhanced query processing for stock-related questions.
